# HR Analytics with Job Postings Data

In [`eda.ipynb`](./eda.ipynb) we explored the data jobs dataset row-by-row: how many postings exist, what fields are missing, what skills are most common. In this notebook we put on a different hat. We act as a **People Analytics / HR Analytics team** at a fictional employer and ask the kinds of questions that drive real talent decisions.

## What is HR analytics?

HR analytics (also called _people analytics_ or _workforce analytics_) is the discipline of using data to make better decisions about an organization's people. Modern HR teams routinely use external labor market data, exactly like our Luke Barousse dataset, for:

- **Competitive compensation benchmarking.** Are we paying our data scientists in Chicago at the 50th percentile of the market, or the 25th? If our offers keep getting declined, the data tells us why.
- **Talent scouting and market mapping.** Where is the supply of senior data engineers concentrated? Which competitors hire the most heavily, and what skills are they screening for?
- **Retention and attrition risk.** When the external market for a skill heats up (more postings, higher salaries, more remote options), internal employees with that skill become flight risks.
- **Skill-gap planning.** What skills are growing fastest in postings? Should we build training programs, or hire ahead of the curve?

We will walk through three of these workflows on the same dataset.

:::{important} Real-world caveats
Job postings are not a perfect mirror of the labor market. They overweight the minority of postings that disclose salary, they reflect _demand_ rather than the actual workforce, and they include reposts. A real HR analytics team would join this data with internal HRIS records, surveys, and offer-acceptance data. The techniques shown here are still the same ones used in practice.
:::


In [1]:
import polars as pl
import plotly.express as px

pl.Config.set_tbl_rows(20)
import plotly.io as pio

pio.templates.default = "simple_white"

## 📥 Load the Cleaned Dataset

We pick up where the EDA notebook left off, loading the preprocessed Parquet file with `job_skills` already parsed into a `List[str]`, `job_posted_date` parsed into a proper datetime, and the mislabeled "Sudan" postings relabeled as United States.

If you have not run [`eda.ipynb`](./eda.ipynb), the file `data_jobs_preprocessed.parquet` does not exist yet. In that case the cell below downloads the raw data and repeats the EDA steps this notebook depends on. It skips parsing `job_type_skills`, which this notebook does not use.

In [2]:
from pathlib import Path

CLEAN_PATH = Path("data_jobs_preprocessed.parquet")
RAW_URL = "https://media.githubusercontent.com/media/bdi593/datasets/main/data-jobs/data_jobs.parquet"

if CLEAN_PATH.exists():
    df = pl.read_parquet(CLEAN_PATH)
else:
    # Same cleaning steps as eda.ipynb, limited to the columns used here.
    df = (
        pl.read_parquet(RAW_URL)
        .filter(pl.col("job_title").is_not_null())
        .with_columns(
            pl.col("job_posted_date").str.to_datetime(),
            pl.col("job_skills")
            .str.replace_all("'", '"')
            .str.json_decode(dtype=pl.List(pl.String)),
            pl.when(pl.col("search_location") == "Sudan")
            .then(pl.lit("United States"))
            .otherwise(pl.col("job_country"))
            .alias("job_country"),
        )
    )

print("Rows:", df.height)
df.head(3)

Rows: 785740


job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,job_no_degree_mention,job_health_insurance,job_country,salary_rate,salary_year_avg,salary_hour_avg,company_name,job_skills,job_type_skills
str,str,str,str,str,bool,str,datetime[μs],bool,bool,str,str,f64,f64,str,list[str],struct[10]
"""Senior Data Engineer""","""Senior Clinical Data Engineer …","""Watertown, CT""","""via Work Nearby""","""Full-time""",false,"""Texas, United States""",2023-06-16 13:44:15,false,false,"""United States""",null,null,null,"""Boehringer Ingelheim""",null,null
"""Data Analyst""","""Data Analyst""","""Guadalajara, Jalisco, Mexico""","""via BeBee México""","""Full-time""",false,"""Mexico""",2023-01-14 13:18:07,false,false,"""Mexico""",null,null,null,"""Hewlett Packard Enterprise""","[""r"", ""python"", … ""tableau""]","{null,[""power bi"", ""tableau""],null,null,null,null,null,null,null,[""r"", ""python"", … ""nosql""]}"
"""Data Engineer""","""Data Engineer/Scientist/Analys…","""Berlin, Germany""","""via LinkedIn""","""Full-time""",false,"""Germany""",2023-10-10 13:14:55,false,false,"""Germany""",null,null,null,"""ALPHA Augmented Services""","[""python"", ""sql"", … ""jenkins""]","{[""azure""],[""dax""],null,null,null,[""docker"", ""kubernetes"", ""jenkins""],null,[""airflow""],null,[""python"", ""sql"", ""c#""]}"


## 🏢 Scenario

Imagine we work for **NorthStar Insights**, a mid-sized U.S. consulting firm. Leadership is planning the next fiscal year and the head of People asks three questions:

1. _Are our salary bands competitive for Data Analyst, Data Scientist, and Data Engineer roles?_
2. _Where should we focus our recruiting?_
3. _Which skills are heating up in the market, and which of our employees are most exposed to being poached?_

Each section below tackles one of those questions.


## 1. Competitive Compensation Benchmarking

When People teams talk about "benchmarking", they almost always mean computing **percentiles** of the disclosed market salary for a given role and location, then comparing those percentiles to the company's own salary bands.

A typical band looks like this:

- **P25**: the bottom of the market. If you target this percentile you are likely hiring entry-level or accepting longer time-to-fill.
- **P50 (median)**: "market rate". A reasonable default target for most roles.
- **P75**: paying for top talent or scarce skills.
- **P90**: premium / executive band.


In [3]:
FOCUS_ROLES = ["Data Analyst", "Data Scientist", "Data Engineer"]

comp_bench = (
    df.filter(
        pl.col("job_country") == "United States",
        pl.col("salary_year_avg").is_not_null(),
        pl.col("job_title_short").is_in(FOCUS_ROLES),
    )
    .group_by("job_title_short")
    .agg(
        n=pl.len(),
        p25=pl.col("salary_year_avg").quantile(0.25),
        p50=pl.col("salary_year_avg").quantile(0.50),
        p75=pl.col("salary_year_avg").quantile(0.75),
        p90=pl.col("salary_year_avg").quantile(0.90),
    )
    .sort("p50", descending=True)
)
comp_bench

job_title_short,n,p25,p50,p75,p90
str,u32,f64,f64,f64,f64
"""Data Scientist""",5118,105000.0,130000.0,162623.5,202500.0
"""Data Engineer""",3416,107500.0,125000.0,150000.0,186500.0
"""Data Analyst""",4473,71239.0,90000.0,112500.0,135000.0


### Comparing NorthStar's bands to the market

Suppose NorthStar's current pay bands (midpoints) are:

| Role           | NorthStar midpoint |
| -------------- | ------------------ |
| Data Analyst   | \$78,000           |
| Data Scientist | \$130,000          |
| Data Engineer  | \$120,000          |

We can join this small table against the market percentiles and show the gap.


In [4]:
northstar_bands = pl.DataFrame(
    {
        "job_title_short": ["Data Analyst", "Data Scientist", "Data Engineer"],
        "northstar_midpoint": [78_000, 130_000, 120_000],
    }
)

gap = (
    comp_bench.join(northstar_bands, on="job_title_short")
    .with_columns(
        gap_vs_p50=(pl.col("northstar_midpoint") - pl.col("p50")).round(0),
        position_in_market=(
            (pl.col("northstar_midpoint") - pl.col("p25"))
            / (pl.col("p75") - pl.col("p25"))
        ).round(2),
    )
    .select(
        "job_title_short",
        "northstar_midpoint",
        "p25",
        "p50",
        "p75",
        "gap_vs_p50",
        "position_in_market",
    )
)
gap

job_title_short,northstar_midpoint,p25,p50,p75,gap_vs_p50,position_in_market
str,i64,f64,f64,f64,f64,f64
"""Data Analyst""",78000,71239.0,90000.0,112500.0,-12000.0,0.16
"""Data Scientist""",130000,105000.0,130000.0,162623.5,0.0,0.43
"""Data Engineer""",120000,107500.0,125000.0,150000.0,-5000.0,0.29


:::{tip} How to read `position_in_market`
We've expressed the company midpoint as its position within the market's interquartile range (P25 to P75). A value of `0.0` means we are paying at P25 and `1.0` means we are at P75. A value of `0.5` is the middle of that range, which equals P50 only when the market is symmetric around the median. For example, NorthStar's Data Scientist midpoint equals the market P50 (\$130,000) but scores `0.43`, because the market's P75 sits further above the median than P25 sits below it. If a role's value is near or below `0.0`, expect declined offers and longer time-to-fill.
:::

## 2. Talent Scouting & Market Mapping

When NorthStar opens a new Data Engineer role, two questions matter immediately:

1. **Where is the supply?** Which cities have the most active Data Engineer postings? Those are likely where we will find candidates too.
2. **Who are we competing with?** Which companies are posting the most for the same role?


### Where is the demand for Data Engineers?


In [5]:
de_us = df.filter(
    pl.col("job_title_short") == "Data Engineer",
    pl.col("job_country") == "United States",
)

# Keep locations of the form "City, ST". This drops "Anywhere" (remote),
# "United States", state names, and multi-location postings ("(+3 others)").
is_city = pl.col("job_location").str.contains(r"^[^,(]+, [A-Z]{2}$")

print(f"U.S. Data Engineer postings: {de_us.height:,}")
print(f"  tied to one city:  {de_us.filter(is_city).height:,}")
print(f"  'Anywhere' (remote): {de_us.filter(pl.col('job_location') == 'Anywhere').height:,}")

de_locations = (
    de_us.filter(is_city)
    .group_by("job_location")
    .agg(n=pl.len())
    .sort(["n", "job_location"], descending=[True, False])
    .head(15)
)
de_locations

U.S. Data Engineer postings: 41,275
  tied to one city:  31,204
  'Anywhere' (remote): 7,408


job_location,n
str,u32
"""New York, NY""",1401
"""Dallas, TX""",996
"""Atlanta, GA""",882
"""Chicago, IL""",838
"""Austin, TX""",746
"""Charlotte, NC""",681
"""Washington, DC""",681
"""Houston, TX""",609
"""Boston, MA""",591


In [6]:
fig = px.bar(
    de_locations.to_pandas(),
    x="n",
    y="job_location",
    orientation="h",
    title="Top 15 U.S. cities by Data Engineer posting volume",
    template="simple_white",
    height=600,
)
fig.update_yaxes(categoryorder="total ascending")
fig.show()

Among postings tied to a single city, New York, Dallas, and Atlanta have the most Data Engineer postings, followed by Chicago and Austin. These are natural places for hub-based roles. Keep three caveats in mind:

- About 18% of U.S. Data Engineer postings (7,408 of 41,275) are listed as "Anywhere" and are not tied to a city, so remote hiring is a large channel of its own.
- Posting volume measures employer demand. It suggests, but does not prove, where experienced candidates live.
- The U.S. postings come from seven searches: New York, California, Illinois, Georgia, Texas, Florida, and the relabeled "Sudan" search, which returned many Texas and New Mexico jobs. Cities in those states may be over-represented compared with cities elsewhere.

### Who are the most active competitors?


In [7]:
# Job boards, staffing agencies, and freelance marketplaces post on behalf of
# other employers, so they are not direct competitors for talent. This list was
# built by hand from the names at the top of the ranking. It is a judgment call
# and not complete, so check the names in your own ranking.
INTERMEDIARIES = [
    "Dice",
    "Insight Global",
    "Upwork",
    "Robert Half",
    "Motion Recruitment",
    "Diverse Lynx",
    "Jobot",
    "Jobs for Humanity",
    "CyberCoders",
    "Apex Systems",
    "TEKsystems",
    "The Judge Group",
    "Kforce",
    "Harnham",
    "ClearanceJobs",
    "Compunnel Inc.",
]

top_de_employers = (
    de_us.with_columns(
        # The same firm can appear under two names.
        pl.col("company_name").replace({"KPMG US": "KPMG"})
    )
    .filter(
        ~pl.col("company_name").is_in(INTERMEDIARIES),
        ~pl.col("company_name").str.starts_with("Get It Recruit"),
        ~pl.col("company_name").str.starts_with("TechFetch"),
    )
    .group_by("company_name")
    .agg(n=pl.len())
    .sort(["n", "company_name"], descending=[True, False])
    .head(10)
)
top_de_employers

company_name,n
str,u32
"""Capital One""",763
"""KPMG""",554
"""Booz Allen Hamilton""",492
"""Deloitte""",238
"""CVS Health""",183
"""Wells Fargo""",183
"""Fidelity Investments""",182
"""Tesla""",168
"""Guidehouse""",157


Before filtering, the top of this ranking was dominated by job boards and staffing agencies (Dice, Insight Global, Upwork, Robert Half), and KPMG appeared twice under two names. After removing intermediaries and merging the two KPMG names, the most active Data Engineer employers are banks and financial firms (Capital One, Wells Fargo, Fidelity Investments), consulting firms (KPMG, Booz Allen Hamilton, Deloitte, Guidehouse), and a few others (CVS Health, Tesla, Meta). As a consulting firm, NorthStar competes most directly with the consulting names on this list. The list is also a reasonable starting point for outbound recruiting, since people at these companies do similar work.

### Remote vs on-site mix

If we insist on on-site, we shrink the addressable pool. The data tells us how much.


In [8]:
remote_mix = (
    df.filter(
        pl.col("job_title_short").is_in(FOCUS_ROLES),
        pl.col("job_country") == "United States",
    )
    .group_by("job_title_short")
    .agg(
        n=pl.len(),
        remote_share=pl.col("job_work_from_home").mean().round(3),
    )
    .sort("job_title_short")
)
remote_mix

job_title_short,n,remote_share
str,u32,f64
"""Data Analyst""",69600,0.077
"""Data Engineer""",41275,0.179
"""Data Scientist""",67374,0.115


About 18% of U.S. Data Engineer postings are remote, compared with about 12% for Data Scientists and 8% for Data Analysts. If NorthStar requires Data Engineers to work on-site, candidates who want remote work have plenty of alternatives: nearly one U.S. Data Engineer posting in five is remote.

## 3. Skill-Demand Heatmap & Retention Risk

When the _external_ demand for a skill heats up, _internal_ employees with that skill can become flight risks: their LinkedIn inboxes fill up, recruiters call them more often, and competing offers get more aggressive.

We can track this by measuring, month by month, the share of postings that mention each skill. The data covers 2023 only, so we cannot compute a year-over-year change. Instead, we compare the second half of the year with the first half, and then draw a heatmap of the monthly shares.

First, count each skill's mentions per month and divide by the number of postings with skills that month.

In [9]:
skill_by_month = (
    df.filter(pl.col("job_skills").is_not_null())
    .with_columns(pl.col("job_posted_date").dt.truncate("1mo").alias("month"))
    .with_columns(pl.len().over("month").alias("postings_in_month"))
    .select("month", "postings_in_month", "job_skills")
    .explode("job_skills")
    .group_by(["month", "job_skills"])
    .agg(n=pl.len(), postings_in_month=pl.col("postings_in_month").first())
    .with_columns(share=pl.col("n") / pl.col("postings_in_month"))
    .sort(["job_skills", "month"])
)
skill_by_month.head(12)

month,job_skills,n,postings_in_month,share
datetime[μs],str,u32,u32,f64
2023-01-01 00:00:00,"""airflow""",5136,79039,0.064981
2023-02-01 00:00:00,"""airflow""",3709,55818,0.066448
2023-03-01 00:00:00,"""airflow""",3876,55886,0.069355
2023-04-01 00:00:00,"""airflow""",3805,54186,0.070221
2023-05-01 00:00:00,"""airflow""",3480,45438,0.076588
2023-06-01 00:00:00,"""airflow""",3810,53564,0.07113
2023-07-01 00:00:00,"""airflow""",3685,54630,0.067454
2023-08-01 00:00:00,"""airflow""",4035,63695,0.063349
2023-09-01 00:00:00,"""airflow""",3289,51112,0.064349


### Compare the last six months with the six months before

For each skill, we compute the share of postings mentioning it in the latest 180 days (roughly July to December 2023) and in the 180 days before that (roughly January to early July 2023). We keep only skills mentioned in at least 1% of postings in the earlier window. Without that filter, the biggest percentage changes come from niche skills that appear in one or two postings per thousand, which say little about the people we employ.

`change_pp` is the change in percentage points, and `change_pct` is the relative change.

In [10]:
from datetime import timedelta

last_date = df.select(pl.col("job_posted_date").max()).item()
end_recent = last_date + timedelta(seconds=1)  # include the last posting
start_recent = end_recent - timedelta(days=180)  # about six months
start_prior = start_recent - timedelta(days=180)  # the six months before that


def skill_share(start, end):
    window = df.filter(
        pl.col("job_skills").is_not_null(),
        pl.col("job_posted_date") >= start,
        pl.col("job_posted_date") < end,
    )
    total = window.height
    return (
        window.select("job_skills")
        .explode("job_skills")
        .group_by("job_skills")
        .agg(n=pl.len())
        .with_columns(share=(pl.col("n") / total))
    )


recent = skill_share(start_recent, end_recent).rename(
    {"n": "n_recent", "share": "share_recent"}
)
prior = skill_share(start_prior, start_recent).rename(
    {"n": "n_prior", "share": "share_prior"}
)

trend = (
    recent.join(prior, on="job_skills", how="inner")
    .filter(pl.col("share_prior") >= 0.01)
    .with_columns(
        pct_prior=(pl.col("share_prior") * 100).round(2),
        pct_recent=(pl.col("share_recent") * 100).round(2),
        change_pp=((pl.col("share_recent") - pl.col("share_prior")) * 100).round(2),
        change_pct=(
            (pl.col("share_recent") - pl.col("share_prior"))
            / pl.col("share_prior")
            * 100
        ).round(1),
    )
    .sort(["change_pct", "job_skills"], descending=[True, False])
    .select("job_skills", "pct_prior", "pct_recent", "change_pp", "change_pct")
)
print(f"Skills mentioned in at least 1% of postings: {trend.height}")
trend.head(10)

Skills mentioned in at least 1% of postings: 81


job_skills,pct_prior,pct_recent,change_pp,change_pct
str,f64,f64,f64,f64
"""pytorch""",3.79,4.4,0.61,16.1
"""matplotlib""",1.25,1.42,0.17,13.5
"""scikit-learn""",2.88,3.13,0.25,8.7
"""tensorflow""",4.8,5.13,0.32,6.7
"""react""",1.34,1.42,0.08,6.0
"""html""",1.16,1.21,0.05,4.4
"""spring""",1.06,1.11,0.05,4.3
"""c""",2.31,2.39,0.08,3.6
"""power bi""",14.46,14.98,0.51,3.6


And the skills whose share fell the most:

In [11]:
trend.tail(10).sort(["change_pct", "job_skills"])

job_skills,pct_prior,pct_recent,change_pp,change_pct
str,f64,f64,f64,f64
"""cassandra""",1.99,1.7,-0.29,-14.7
"""qlik""",2.73,2.33,-0.4,-14.7
"""sas""",13.37,11.5,-1.86,-13.9
"""elasticsearch""",1.59,1.37,-0.22,-13.8
"""powerpoint""",4.56,3.96,-0.6,-13.1
"""kafka""",7.96,6.93,-1.03,-12.9
"""alteryx""",1.96,1.71,-0.25,-12.7
"""mongodb""",5.78,5.07,-0.71,-12.3
"""hadoop""",10.3,9.04,-1.26,-12.2


### A heatmap of monthly demand

The heatmap shows the five fastest-rising and five fastest-falling skills from the tables above. Each cell is the skill's share of postings that month, indexed to the skill's own 2023 average (100 = average month). Values above 100 (red) mean the skill was mentioned more often than usual that month.

In [12]:
watch_list = pl.concat([trend.head(5), trend.tail(5)])["job_skills"].to_list()

heat = (
    skill_by_month.filter(pl.col("job_skills").is_in(watch_list))
    .with_columns(
        index=(pl.col("share") / pl.col("share").mean().over("job_skills") * 100).round(0),
        month_label=pl.col("month").dt.strftime("%b"),
    )
    .to_pandas()
    .pivot(index="job_skills", columns="month_label", values="index")
)
month_order = (
    skill_by_month.select(pl.col("month").unique().sort().dt.strftime("%b"))
    .to_series()
    .to_list()
)
heat = heat.loc[watch_list, month_order]

fig = px.imshow(
    heat,
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=100,
    text_auto=True,
    aspect="auto",
    labels={"x": "Month (2023)", "y": "Skill", "color": "Index"},
    title="Monthly share of postings, indexed to each skill's 2023 average",
    height=500,
)
fig.show()

The heatmap shows whether a change is a steady trend or the result of one or two unusual months. PyTorch rises fairly steadily through the year, while SAS, PowerPoint, and Elasticsearch fall fairly steadily. React jumps around from month to month, so its rise is less convincing. Because we picked these ten skills for having the largest changes, some of the pattern is expected by construction. The useful information is how steady it is.

### Translating the heatmap into retention actions

The machine-learning stack (PyTorch, scikit-learn, TensorFlow, Matplotlib) gained share in the second half of 2023, while SAS, Hadoop, Kafka, and several database tools (Cassandra, MongoDB, Elasticsearch) lost share. The changes are small in absolute terms: PyTorch went from about 3.8% to 4.4% of postings. With a single year of data, a half-year comparison can also pick up seasonal effects and changes in what the data collector searched. Treat the rising list as a watch list, not a verdict.

With that caution, the People team can:

1. Pull the internal employee list and flag employees whose primary skill is on the rising list. They are **candidates for a closer look**, not confirmed flight risks.
2. Prioritise _stay interviews_ and proactive compensation reviews for those employees, starting with skills that are both rising and common (PyTorch, TensorFlow).
3. Build a 12-month skills-development roadmap so that _internal mobility_, not external hiring, becomes the first line of defence.

:::{seealso} What if a skill is _cooling off_?
Skills with strongly negative change, such as SAS, Hadoop, and Kafka in the table above, are also strategically interesting: they signal where market demand may be waning and where the company could slow down hiring. Employees with those skills are good candidates for re-skilling investments.
:::

## 📌 Recap

We took a single labor-market dataset and used it to mimic three distinct HR analytics workflows:

| Workflow                         | Question answered                                         | Tools used                                          |
| -------------------------------- | --------------------------------------------------------- | --------------------------------------------------- |
| Compensation benchmarking        | _Are our salary bands competitive?_                       | quantile aggregations, joins to internal data       |
| Talent scouting / market mapping | _Where do we hire and who do we compete with?_            | filtering, group-by, value counts                   |
| Skill demand & retention risk    | _Which skills are gaining or losing share in postings?_   | time-windowed `explode` + group-by, heatmap         |

What we found for NorthStar:

- **Compensation:** the Data Analyst midpoint (\$78,000) is \$12,000 below the U.S. market median and near the bottom of the market's interquartile range. The Data Scientist midpoint matches the median, and the Data Engineer midpoint is \$5,000 below it.
- **Sourcing:** New York, Dallas, and Atlanta have the most city-specific Data Engineer postings, and about 18% of U.S. Data Engineer postings are remote. The most active direct employers are banks and consulting firms.
- **Skills:** the machine-learning stack gained a little share of postings in the second half of 2023, while SAS, Hadoop, and Kafka lost share. The changes are small, so they are a watch list rather than a forecast.

The same dataset supports all three, with nothing new ingested. That is the recurring pattern in real HR analytics: pick a clean source of external market signal, and use it as a lens for compensation, sourcing, and retention decisions.